In [ ]:
# -------------------------------------------------------------------
# APLIKASI NEURAL NETWORK DENGAN UPLOAD FOTO
# Tema: Pengenalan Objek (CIFAR-10)
# Library: TensorFlow (Keras), OpenCV (cv2), Matplotlib
# Dibuat untuk dijalankan di Google Colab
# -------------------------------------------------------------------

# 1. IMPORT LIBRARY YANG DIBUTUHKAN
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files # Untuk fungsi upload di Colab
import cv2 # Untuk pemrosesan gambar

print(f"TensorFlow Versi: {tf.__version__}")

# -------------------------------------------------------------------
# 2. MEMUAT DAN MEMPERSIAPKAN DATASET CIFAR-10
# -------------------------------------------------------------------

# Tentukan nama kelas untuk CIFAR-10
# Ini akan kita gunakan nanti untuk menampilkan hasil prediksi
class_names = ['pesawat', 'mobil', 'burung', 'kucing', 'rusa',
               'anjing', 'katak', 'kuda', 'kapal', 'truk']

# Muat dataset CIFAR-10
print("\nMemuat dataset CIFAR-10...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalisasi data: ubah nilai piksel dari 0-255 menjadi 0-1
x_train = x_train / 255.0
x_test = x_test / 255.0

# y_train dan y_test adalah array 2D, kita biarkan seperti itu
# karena kita akan menggunakan 'sparse_categorical_crossentropy'

print(f"Bentuk data latih (fitur): {x_train.shape}")
print(f"Bentuk data tes (fitur): {x_test.shape}")
print(f"Contoh label: {y_train[0]}") # Harusnya array misal [6] yang artinya 'katak'

# -------------------------------------------------------------------
# 3. MEMBUAT MODEL NEURAL NETWORK (CNN untuk Gambar Berwarna)
# -------------------------------------------------------------------

print("\nMembangun Model Convolutional Neural Network (CNN)...")
model = tf.keras.models.Sequential([
    # Lapisan Konvolusi Pertama
    # input_shape sekarang (32, 32, 3) karena gambar 32x32 dengan 3 channel warna
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Lapisan Konvolusi Kedua
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Lapisan Konvolusi Ketiga (Opsional, untuk akurasi lebih baik)
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),

    # Meratakan output dari lapisan konvolusi
    tf.keras.layers.Flatten(),

    # Lapisan Tersembunyi (Dense Layer)
    tf.keras.layers.Dense(64, activation='relu'),

    # Lapisan Output
    # 10 neuron karena ada 10 kelas
    tf.keras.layers.Dense(10, activation='softmax')
])

# Menampilkan ringkasan arsitektur model
print("\nArsitektur Model CNN:")
model.summary()

# -------------------------------------------------------------------
# 4. KOMPILASI MODEL
# -------------------------------------------------------------------

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy', # Cocok untuk label integer
              metrics=['accuracy'])

print("\nModel berhasil dikompilasi.")

# -------------------------------------------------------------------
# 5. MELATIH MODEL (TRAINING)
# -------------------------------------------------------------------

print("\nMemulai proses pelatihan model...")
# Melatih model. 'epochs=15' mungkin butuh waktu sedikit lebih lama dari MNIST
# (sekitar 2-5 menit di Colab)
history = model.fit(
    x_train,
    y_train,
    epochs=15, # Kita tambah epoch karena datanya lebih kompleks
    validation_data=(x_test, y_test),
    verbose=1
)

print("\nPelatihan model selesai.")

# -------------------------------------------------------------------
# 6. EVALUASI PERFORMA APLIKASI
# -------------------------------------------------------------------

print("\nEvaluasi model pada data tes:")
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Loss (Kerugian) pada data tes: {loss:.4f}")
print(f"Accuracy (Akurasi) pada data tes: {accuracy * 100:.2f}%")
# Catatan: Akurasi CIFAR-10 dengan model sederhana ini mungkin sekitar 65-75%,
# ini normal karena masalahnya jauh lebih sulit daripada MNIST.

# -------------------------------------------------------------------
# 7. FUNGSI UNTUK MENGUPLOAD DAN MEMPREDIKSI GAMBAR
# -------------------------------------------------------------------

def predict_uploaded_image():
    print("\nSilakan upload gambar (misal: foto kucing, mobil, pesawat).")

    # Meminta pengguna untuk mengupload file
    uploaded = files.upload()

    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')

        # Membaca gambar yang diupload
        img_bytes = uploaded[fn]
        img_np = np.frombuffer(img_bytes, np.uint8)
        # Baca sebagai gambar berwarna (IMREAD_COLOR)
        img = cv2.imdecode(img_np, cv2.IMREAD_COLOR)

        if img is None:
            print(f"Gagal membaca gambar {fn}. Pastikan itu adalah file gambar yang valid.")
            continue

        # Pra-pemrosesan gambar:
        # 1. Ubah ukuran ke 32x32 piksel
        img_resized = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA)

        # 2. Konversi BGR (OpenCV) ke RGB (TensorFlow/Matplotlib)
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

        # 3. Normalisasi (skala dari 0-255 ke 0-1)
        img_normalized = img_rgb / 255.0

        # 4. Reshape untuk model: (1, 32, 32, 3) -> 1 gambar, 32x32, 3 channel
        img_final = img_normalized.reshape(1, 32, 32, 3)

        # Menampilkan gambar yang diproses
        plt.imshow(img_final.reshape(32, 32, 3)) # Tampilkan gambar RGB
        plt.title(f"Gambar yang Diupload dan Diproses ({fn})")
        plt.axis('off')
        plt.show()

        # Lakukan prediksi
        prediction_probabilities = model.predict(img_final)
        predicted_class_index = np.argmax(prediction_probabilities)
        predicted_class_name = class_names[predicted_class_index]
        confidence = np.max(prediction_probabilities) * 100

        print(f"\nPrediksi model untuk '{fn}':")
        print(f"Kelas yang diprediksi: {predicted_class_index} -> {predicted_class_name.upper()}")
        print(f"Keyakinan (Confidence): {confidence:.2f}%")

        # Tampilkan 3 prediksi teratas
        print("\nPrediksi teratas:")
        top_3_indices = prediction_probabilities[0].argsort()[-3:][::-1]
        for i in top_3_indices:
            print(f"  - {class_names[i]}: {prediction_probabilities[0][i]*100:.2f}%")
        print("\n" + "="*30 + "\n")

# -------------------------------------------------------------------
# JALANKAN FUNGSI UPLOAD DAN PREDIKSI
# -------------------------------------------------------------------
print("\n--- APLIKASI SIAP DIGUNAKAN ---")
print("Catatan: Model ini dilatih pada gambar 32x32 piksel.")
print("Prediksi pada gambar dunia nyata mungkin tidak selalu akurat,")
print("tetapi ini adalah latihan yang bagus!")

while True:
    try:
        predict_uploaded_image()
    except Exception as e:
        print(f"Terjadi error: {e}")
        break

    # Beri opsi untuk berhenti atau lanjut
    cont = input("Upload gambar lain? (y/n): ")
    if cont.lower() != 'y':
        print("\nTerima kasih telah menggunakan aplikasi ini!")
        break